In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

df = pd.read_csv("../data/q2_customers.csv")

print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

df.head()

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

scaled_df = pd.DataFrame(scaled_data, columns=df.columns)
scaled_df.head()

Scaling is essential before applying K-Means because K-Means uses distance-based calculations. Features with larger numeric ranges, such as annual_spend, could dominate the clustering result if the data is not scaled. StandardScaler puts all features on a similar scale, allowing each feature to contribute fairly.

In [ ]:
wcss = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_df)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1, 11), wcss, marker="o")
plt.title("Elbow Method for Choosing K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.xticks(range(1, 11))
plt.show()

The elbow method helps identify the optimal number of clusters by plotting WCSS for different K values. The optimal K is chosen where the decrease in WCSS begins to slow down. Based on the elbow point in the graph, K = 3 appears to be a suitable choice.

In [ ]:
optimal_k = 3

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaled_df)

df["cluster"] = clusters

df.head()

In [ ]:
centroids_scaled = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=df.drop("cluster", axis=1).columns
)

centroids_original = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=df.drop("cluster", axis=1).columns
)

centroids_original

In [ ]:
df.groupby("cluster").mean()

The cluster centroids show the average customer profile for each cluster. For example, one cluster may represent high-spending customers with frequent visits, another may represent low-spending infrequent customers, and another may represent moderate customers. These segments can help the business target promotions and marketing campaigns more effectively.

In [ ]:
pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_df)

pca_df = pd.DataFrame(pca_data, columns=["PC1", "PC2"])
pca_df["cluster"] = df["cluster"]

pca_df.head()

In [ ]:
explained_variance = pd.DataFrame({
    "Principal Component": ["PC1", "PC2"],
    "Explained Variance Ratio": pca.explained_variance_ratio_
})

explained_variance

In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    columns=scaled_df.columns,
    index=["PC1", "PC2"]
)

loadings

The explained variance ratio shows how much information is captured by each principal component. The feature loadings indicate which original features contribute most strongly to PC1 and PC2. PC1 appears to capture overall customer value or activity, while PC2 captures differences in customer behaviour patterns such as recency or shopping frequency.

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="cluster",
    palette="Set2"
)

plt.title("Customer Segments Visualised Using PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()